# Lab 1 — Supervised Fine-Tuning: contract review that cites its evidence

## Notebook 3 — Deploy to a SageMaker real-time endpoint

Two deployment options, and they suit different demand shapes:

| | **SageMaker real-time endpoint** | **Bedrock Custom Model Import** |
|---|---|---|
| | **SageMaker real-time endpoint** | **Bedrock Custom Model Import** |
| infrastructure | dedicated GPU instance you choose | fully serverless |
| billing | per instance-hour, 24/7 while it exists | per Custom Model Unit in 5-minute active windows |
| idle cost | **you keep paying** | scales to zero |
| cold start | none once InService | tens of seconds after idle |
| best for | steady high throughput | spiky or low-volume workloads |

Deploy
whichever matches how you would serve the model.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket_name}")
print(f"sagemaker session region: {sess.boto_region_name}")

### Find the trained model

This is the one place where the Training-job flow differs structurally from the
serverless one, so it is worth reading rather than running.

Serverless customization registers its result as a **Model Package** in the SageMaker
model registry, and the serverless version of this notebook looks that up by name. A
`ModelTrainer` job registers nothing — it uploads an artifact to S3 and records the URI on
the job. So there is no registry to query, and the job itself is the source of truth:
`DescribeTrainingJob` returns `ModelArtifacts.S3ModelArtifacts`.

Two consequences follow, and both change cells further down:

- **The artifact is a single `model.tar.gz`, not a directory.** SageMaker gzips whatever
  `/opt/ml/model` contained. Mounting it therefore needs `S3DataType="S3Object"` and
  `CompressionType="Gzip"` — the serverless notebook used `S3Prefix`/`None` because
  customization leaves an uncompressed prefix behind.
- **Nothing can be added beside it.** The serverless notebook uploads a vLLM config and a
  reasoning-parser plugin into the model prefix so the container picks them up from
  `/opt/ml/model/`. You cannot add files to a tarball that way, so this notebook
  configures vLLM entirely through `SM_VLLM_*` environment variables instead.

The lookup below takes the most recent **Completed** job whose name starts with
`TRAIN_JOB_PREFIX` from `config.py` — the same constant notebook 2 used as
`base_job_name`, so the two cannot drift. Override `TRAINING_JOB_NAME` to deploy a
specific run.


In [ ]:
from config import BASE_MODEL_ID, MODEL_SLUG, TRAIN_JOB_PREFIX

# Set this to a specific job name to deploy an earlier run instead of the latest.
TRAINING_JOB_NAME = None

if TRAINING_JOB_NAME is None:
    jobs = sm_client.list_training_jobs(
        NameContains=TRAIN_JOB_PREFIX,
        StatusEquals="Completed",
        SortBy="CreationTime",
        SortOrder="Descending",
        MaxResults=10,
    )["TrainingJobSummaries"]
    assert jobs, (
        f"no Completed training job whose name contains {TRAIN_JOB_PREFIX!r} - run "
        "notebook 2 first, and check it reached Completed rather than Failed")
    TRAINING_JOB_NAME = jobs[0]["TrainingJobName"]
    if len(jobs) > 1:
        print(f"{len(jobs)} completed runs found; using the newest:")

job = sm_client.describe_training_job(TrainingJobName=TRAINING_JOB_NAME)
model_data_url = job["ModelArtifacts"]["S3ModelArtifacts"]

print(f"training job:  {TRAINING_JOB_NAME}")
print(f"finished:      {job['TrainingEndTime']:%Y-%m-%d %H:%M}")
print(f"base model:    {BASE_MODEL_ID}")
print(f"artifact:      {model_data_url}")

# The artifact is the *merged* checkpoint - base weights with the LoRA adapter already
# applied, because notebook 2's recipe sets `merge_weights: true` and
# `output_dir: /opt/ml/model`. vLLM can serve it directly; there is no adapter to apply
# at load time. Confirm the size, since a suspiciously small tarball means the merge did
# not run and only the adapter was saved.
key = model_data_url.split(f"{bucket_name}/", 1)[1]
size_gb = s3_client.head_object(Bucket=bucket_name, Key=key)["ContentLength"] / 1024**3
print(f"size:          {size_gb:.1f} GB", end="  ")
print("(merged checkpoint)" if size_gb > 3 else "!! too small - adapter only?")


### Resource names

In [ ]:
import hashlib

MAX_NAME = 63


def rname(base, suffix):
    cand = f"{base}{suffix}"
    if len(cand) <= MAX_NAME:
        return cand
    digest = hashlib.sha1(base.encode()).hexdigest()[:6]
    keep = MAX_NAME - len(suffix) - len(digest) - 1
    return f"{base[:keep].rstrip('-')}-{digest}{suffix}"


# MODEL_SLUG, not BASE_MODEL_ID: a Hugging Face id carries an org prefix, and every
# name below has to match SageMaker's ([a-zA-Z0-9]([a-zA-Z0-9-]){0,62})(?<!-) - the
# "/" alone is rejected. config.py derives the slug once so all four names agree.
stem = f"{MODEL_SLUG}-contractnli"
model_name = rname(stem, "-sft-m")
endpoint_config_name = rname(stem, "-sft-cfg")
endpoint_name = rname(stem, "-sft-ep")
print(model_name, endpoint_config_name, endpoint_name, sep="\n")


#### Instance type

In [ ]:
instance_count = 1

instance_type = "ml.g5.xlarge"
number_of_gpu = 1  # -> vLLM tensor parallel size

FAST_FAIL = True

health_check_timeout = 300 if FAST_FAIL else 1200

# The download either finishes or it does not - a broken container does not consume this -
# so it stays generous regardless. 6.2 GB needs to land inside it.
model_download_timeout = 1200

print(f"{instance_type}: {number_of_gpu} x A10G, {instance_count} instance(s)")


#### vLLM configuration

Everything vLLM needs, passed as environment variables on the model. The container maps
`SM_VLLM_FOO_BAR` to vLLM's `--foo-bar`.

`SM_VLLM_SERVED_MODEL_NAME` is the one to notice. vLLM's OpenAI-compatible API validates the
`"model"` field of each request against the name it served the weights under, which defaults
to the model *path* — `/opt/ml/model` here. Setting it explicitly gives requests a readable
name, and the smoke test below sends exactly this value.


In [ ]:
import json

SERVED_MODEL_NAME = "nemotron-contractnli"

env = {
    "SM_VLLM_MODEL": "/opt/ml/model",  # where SageMaker unpacks the artifact
    "SM_VLLM_SERVED_MODEL_NAME": SERVED_MODEL_NAME,
    "SM_VLLM_DTYPE": "bfloat16",
    "SM_VLLM_GPU_MEMORY_UTILIZATION": "0.9",
    "SM_VLLM_MAX_MODEL_LEN": json.dumps(1024 * 16),
    "SM_VLLM_MAX_NUM_SEQS": "16",
    "SM_VLLM_ENABLE_CHUNKED_PREFILL": "true",
    "SM_VLLM_KV_CACHE_DTYPE": "auto",
    "SM_VLLM_TENSOR_PARALLEL_SIZE": str(number_of_gpu),
}

for k, v in env.items():
    print(f"  {k} = {v}")


### The serving container

**The container's transformers version has to match the one that trained the model.** That
sounds like an implementation detail; it is the single thing most likely to break this
notebook, and it fails late and confusingly.

`save_pretrained` writes a `config.json` in the vocabulary of whatever transformers wrote
it. For this architecture, transformers 5.x records the per-layer mix as
`layer_types: ["full_attention", "linear_attention", ...]`, while older versions expect
`{"attention", "mamba", "mlp", "moe"}`. Serve a 5.x checkpoint on a container with an older
transformers and vLLM dies before it looks at a single weight:

```
ValueError: `layers_block_type` contains invalid types: {'full_attention', 'linear_attention'}.
            Must be one of: {'attention', 'moe', 'mamba', 'mlp'}
```

NVIDIA's published `config.json` sidesteps this by carrying no `layer_types` at all — only
`hybrid_override_pattern` — so the base model loads anywhere. Your fine-tuned copy does not
have that luxury: it was re-serialised by the training container.

That is why this cell uses the **`huggingface-vllm`** image rather than the plain `vllm`
one. Its tags name the bundled transformers version explicitly, so the pairing with
`scripts/requirements.txt` is checkable before you launch instead of discoverable from a
crash loop:

| vLLM | transformers | matches our training pin (5.15.1)? |
|---|---|---|
| **0.28.0** | **5.15.0** | **yes** |
| 0.27.1 / 0.26.0 / 0.25.1 / 0.22.1 | 5.10.2 | no |
| 0.21.0 | 5.8.1 | no |
| 0.17.0 | 4.57.5 | no |

Full list: [available DLC images](https://aws.github.io/deep-learning-containers/reference/available_images/#huggingface-vllm-inference).

**If you bump `transformers` in `scripts/requirements.txt`, revisit this cell.** The assert
below is there to make that coupling fail loudly rather than 20 minutes into a deployment.


In [ ]:
region = sess.boto_region_name

VLLM_VERSION = "0.28.0"
CONTAINER_VERSION = f"vllm:{VLLM_VERSION}-gpu-py312-cu130-ubuntu24.04-sagemaker"

inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/{CONTAINER_VERSION}"

print(inference_image)


### Create the model

The `Model` resource pairs the container image with the artifact and the environment above.
It has to exist **before** the endpoint config, because the config now names it directly.

Note the data source: a Training job's output is a single gzipped tarball, so this is an
`S3Object` with `Gzip` compression, and SageMaker unpacks it into `/opt/ml/model`. The
serverless lab mounts an uncompressed `S3Prefix` instead, because customization leaves a
directory rather than an archive.


In [ ]:
from sagemaker.core.resources import Model
from sagemaker.core.shapes import (ContainerDefinition, ModelDataSource,
                                   S3ModelDataSource)

try:
    Model.get(model_name)
    print(f"model exists: {model_name}")
except Exception:
    Model.create(
        model_name=model_name,
        primary_container=ContainerDefinition(
            image=inference_image,
            model_data_source=ModelDataSource(
                s3_data_source=S3ModelDataSource(
                    s3_uri=model_data_url, s3_data_type="S3Object",
                    compression_type="Gzip")),
            environment=env),
        execution_role_arn=role)
    print(f"created model: {model_name}")


### Create the endpoint config

This is where the direct deployment happens: `model_name` on the `ProductionVariant`. With
an inference component that field is left unset and the model is attached afterwards; here
the variant owns the model, so creating the endpoint is the last step.

An endpoint config is **immutable**, and its name is derived from the model id — so after
changing `instance_type` a get-first check would silently reuse the config built for the old
one. The cell verifies the instance type instead of assuming it.


In [ ]:
from sagemaker.core.resources import Endpoint, EndpointConfig
from sagemaker.core.shapes import ProductionVariant

existing = None
try:
    existing = EndpointConfig.get(endpoint_config_name)
except Exception:
    pass

if existing is not None:
    variant = existing.production_variants[0]
    if variant.instance_type != instance_type or variant.model_name != model_name:
        raise SystemExit(
            f"endpoint config {endpoint_config_name} already exists with "
            f"instance_type={variant.instance_type}, model_name={variant.model_name}, "
            f"but this notebook wants {instance_type} / {model_name}. An endpoint config "
            "is immutable: delete the endpoint, then the config (cleanup cells at the "
            "end), before re-running this cell.")
    print(f"endpoint config exists and matches: {endpoint_config_name}")
else:
    EndpointConfig.create(
        endpoint_config_name=endpoint_config_name,
        production_variants=[ProductionVariant(
            variant_name="AllTraffic",
            model_name=model_name,
            instance_type=instance_type,
            initial_instance_count=instance_count,
            initial_variant_weight=1.0,
            model_data_download_timeout_in_seconds=model_download_timeout,
            container_startup_health_check_timeout_in_seconds=health_check_timeout,
            inference_ami_version="al2-ami-sagemaker-inference-gpu-3-1",
            routing_config={"routing_strategy": "LEAST_OUTSTANDING_REQUESTS"})])
    print(f"created endpoint config: {endpoint_config_name} ({instance_type})")


### Create the endpoint

Provisioning the instance, pulling the vLLM container, downloading and unpacking the
artifact, and loading the weights all happen here, so this is the slow cell — budget
**10-15 minutes**. When it returns `InService` the model is already loaded and ready; there
is no second resource to wait for.


In [ ]:
import re
import time

CRASH_PATTERNS = ("Error", "Traceback", "RuntimeError", "ValidationError",
                  "ValueError", "Engine core initialization failed")
PID_PREFIX = re.compile(r"^\(\w+ pid=\d+\)\s*")


def watch_endpoint(name, poll=20, quiet_timeout=None):
    """Wait for InService, surfacing container errors as soon as they are logged.

    `wait_for_status` only sees the endpoint status, and SageMaker cannot tell a crashing
    container from a slow one - it only knows whether /ping answered. So it waits out
    ContainerStartupHealthCheckTimeoutInSeconds, then replaces the instance and retries,
    which makes the real wait N x that timeout.

    Two things this has to get right, both learned the hard way:

    - **Filter by time.** The log group is named after the endpoint and OUTLIVES it, so a
      redeployed endpoint inherits every line from previous attempts. Without a startTime
      you replay old failures and think the new deployment is broken.
    - **Strip the pid before deduping.** vLLM's supervisor restarts the API server with a
      new pid each time, so the same traceback arrives as hundreds of distinct lines.
    """
    logs = boto3.client("logs", region_name=region)
    group = f"/aws/sagemaker/Endpoints/{name}"

    # Only lines from this deployment. Endpoint.get gives creation_time as a datetime;
    # back off a minute to catch anything logged during provisioning.
    created = Endpoint.get(name).creation_time
    start_ms = int(created.timestamp() * 1000) - 60_000

    seen, started, attempts = set(), time.time(), 0

    while True:
        endpoint = Endpoint.get(name)
        status = endpoint.endpoint_status
        mins = (time.time() - started) / 60

        try:
            events = logs.filter_log_events(
                logGroupName=group,
                startTime=start_ms,
                filterPattern=" ".join(f'?"{p}"' for p in CRASH_PATTERNS),
            )["events"]
            attempts = len({e["logStreamName"] for e in events}) or attempts
            for event in events:
                line = PID_PREFIX.sub("", event["message"].strip())
                if line and line not in seen:
                    seen.add(line)
                    print(f"  LOG {line[:200]}")
        except logs.exceptions.ResourceNotFoundException:
            pass  # container has not logged yet

        note = f"  ({attempts} instance attempt(s), {len(seen)} distinct error line(s))" if seen else ""
        print(f"[{mins:5.1f} min] {status}{note}")

        if status == "InService":
            return endpoint
        if status == "Failed":
            raise RuntimeError(f"endpoint failed: {endpoint.failure_reason}")
        if quiet_timeout and mins > quiet_timeout:
            raise TimeoutError(
                f"still {status} after {mins:.0f} min. SageMaker retries a crashing "
                "container, so this can persist - read the LOG lines above, then delete "
                "the endpoint rather than waiting.")
        time.sleep(poll)


try:
    endpoint = Endpoint.get(endpoint_name)
    print(f"endpoint exists ({endpoint.endpoint_status}): {endpoint_name}")
except Exception:
    endpoint = Endpoint.create(endpoint_name=endpoint_name,
                               endpoint_config_name=endpoint_config_name)
    print(f"creating endpoint: {endpoint_name}")

# quiet_timeout stops *this cell*, not the deployment. If it trips, the endpoint is still
# Creating and still billing - delete it.
endpoint = watch_endpoint(endpoint_name, quiet_timeout=25)
print(f"\nready. endpoint InService: {endpoint_name}")


### Smoke test on a real contract

One request, to prove the endpoint serves what notebook 2 trained.

The request is built by `C.build_messages(doc, labels)` and carries
`C.CHAT_TEMPLATE_KWARGS`, both imported from `contractnli.py` rather than rebuilt here.
That matters more than it looks: those same two values produced the `prompt` column of
every training record, and the chat template that renders them into the string the model
reads is the model’s own — applied by TRL inside the training job, and by vLLM inside this
container. So there is no train/serve skew to reason about, which is the whole reason the
prompt lives in one module.

What to look for: valid JSON, and **all 17** checklist items. A truncated answer usually
means `enable_thinking` did not reach the template and the model reasoned until it ran out
of tokens — the failure the flag exists to prevent.

No accuracy is measured here. That is notebook 4’s job, over the whole held-out split.


In [ ]:
import json
import re
import time

import boto3
from botocore.config import Config

import contractnli as C

C.ensure_dataset("./data")
test_docs, labels = C.load("test")
doc = test_docs[0]

# Raw SageMaker runtime call - this is the request an application would make.
# The vLLM container speaks the OpenAI chat schema.
smr = boto3.client(
    "sagemaker-runtime",
    region_name=region,
    config=Config(read_timeout=300, retries={"total_max_attempts": 3}),
)

def ask_endpoint(doc, max_tokens=4000):
    """Invoke the deployed model and return (text, usage).

    The turns come from `C.build_messages` and the reasoning switch from
    `C.CHAT_TEMPLATE_KWARGS`, so both are imported rather than rebuilt. This is the same
    pair notebook 1 wrote into every training record, rendered by the same chat template -
    the container applies it here, TRL applied it there. That is what makes this a real
    serving check rather than an approximation of one.
    """
    body = {
        # Must match SM_VLLM_SERVED_MODEL_NAME from the env cell: vLLM validates this
        # against the name it served the weights under, and 404s on a mismatch.
        "model": SERVED_MODEL_NAME,
        "messages": [{"role": m["role"],
                      "content": [{"type": "text", "text": m["content"]}]}
                     for m in C.build_messages(doc, labels)],
        # The switch that keeps this model from spending its whole budget inside <think>.
        # Training passed the same dict as a per-record column; vLLM takes it per request
        # and forwards it to apply_chat_template.
        "chat_template_kwargs": C.CHAT_TEMPLATE_KWARGS,
        "max_tokens": max_tokens,
        "temperature": 0.0,
        "stream": False,
    }
    # No InferenceComponentName: this endpoint hosts exactly one model, named by its
    # production variant, so the endpoint name is the whole address.
    response = smr.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Body=json.dumps(body),
    )
    payload = json.loads(response["Body"].read())
    text = payload["choices"][0]["message"]["content"]
    usage = payload.get("usage", {})
    return text, usage


def parse_verdicts(text):
    """Minimal check that the served model returned a usable verdict object."""
    body = re.sub(r"<think>.*?</think>", " ", text or "", flags=re.DOTALL)
    fence = re.search(r"```(?:json)?\s*(.*?)```", body, re.DOTALL)
    if fence:
        body = fence.group(1)
    start, end = body.find("{"), body.rfind("}")
    if start == -1 or end <= start:
        return None
    try:
        return json.loads(body[start:end + 1])
    except json.JSONDecodeError:
        return None


t0 = time.time()
text, usage = ask_endpoint(doc)
elapsed = time.time() - t0

pred = parse_verdicts(text)
ok = pred is not None
print(f"first call {elapsed:.1f}s | valid JSON: {ok} | usage: {usage}")
print(f"answered {len(pred) if ok else 0} of {len(labels)} checklist items")




### Leave the endpoint running

There is no teardown here. Notebook 4 scores the whole held-out split against this
endpoint, so it has to stay up, and the deletes live at the end of that notebook.

> **Important:** a real-time endpoint bills per instance-hour for as long as it exists,
> whether or not you send it traffic. If you stop after this notebook, run the cleanup
> cell at the end of notebook 4 anyway.

Continue to **notebook 4** to measure it.